# Database Setup: Coffee Reviews Schema

Splits the cleaned CSV into two SQLite tables:
- **coffee_reviews**: structured metadata (name, roaster, location, scores, etc.)
- **coffee_review_text**: review text used for embedding/retrieval

Each row in both tables is linked by a shared `review_uid`.

In [9]:
from pathlib import Path

# Project paths
cwd = Path.cwd()
PROJECT_ROOT = cwd.parents[0]
DATA_ROOT = PROJECT_ROOT / "data"
SQL_DATA_ROOT = PROJECT_ROOT / "db"

SQL_DATA_ROOT.mkdir(exist_ok=True, parents=True)

In [10]:
# imports and Configuration
import uuid
import sqlite3
import pandas as pd

In [12]:
# provide file paths
CSV_PATH = DATA_ROOT / "interim" / "01_coffee_review_clean.csv"
SQL_DB_PATH = SQL_DATA_ROOT / "coffee_reviews.db"

## Step 1: Load and Inspect Cleaned CSV

Load the cleaned CSV into a DataFrame and confirm the structure matches
what we expect before doing any transformations.

In [13]:
df = pd.read_csv(CSV_PATH)

print(f"Shape: {df.shape}")
df.info()
df.head()

Shape: (2280, 18)
<class 'pandas.DataFrame'>
RangeIndex: 2280 entries, 0 to 2279
Data columns (total 18 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   slug          2280 non-null   str    
 1   rating        2280 non-null   int64  
 2   roaster       2280 non-null   str    
 3   name          2280 non-null   str    
 4   location      2280 non-null   str    
 5   origin        2280 non-null   str    
 6   roast         2280 non-null   str    
 7   est_price     2280 non-null   str    
 8   review_date   2280 non-null   str    
 9   agtron        2280 non-null   str    
 10  aroma         2253 non-null   float64
 11  acid          1945 non-null   float64
 12  body          2277 non-null   float64
 13  flavor        2277 non-null   float64
 14  aftertaste    2277 non-null   float64
 15  desc_1        2280 non-null   str    
 16  desc_3        2280 non-null   str    
 17  desc_2_clean  2279 non-null   str    
dtypes: float64(5), int64(

,slug,rating,roaster,name,location,origin,roast,est_price,review_date,agtron,aroma,acid,body,flavor,aftertaste,desc_1,desc_3,desc_2_clean
0,https://www.coffeereview.com/review/sweety-esp...,95,A.R.C.,“Sweety” Espresso Blend,"Hong Kong, China",Panama; Ethiopia,Medium-Light,HKD $250/227 grams,2017-11-01,50/73,9.0,NaN,9.0,9.0,9.0,"Evaluated as espresso. Sweet-toned, deeply ric...",A radiant espresso blend that shines equally i...,An espresso blend comprised of coffees from Pa...
1,https://www.coffeereview.com/review/flora-blen...,94,A.R.C.,Flora Blend Espresso,"Hong Kong, China",Africa; Asia Pacific,Medium-Light,HKD $158/227 grams,2017-11-01,54/77,9.0,NaN,9.0,9.0,8.0,"Evaluated as espresso. Sweetly tart, floral-to...","A floral-driven straight shot, amplified with ...",An espresso blend comprised of coffees from Af...
2,https://www.coffeereview.com/review/ethiopia-s...,92,Revel Coffee,Ethiopia Shakiso Mormora,"Billings, Montana","Guji Zone, southern Ethiopia",Medium-Light,$16.00/12 ounces,2017-11-01,54/70,9.0,8.0,8.0,9.0,8.0,"Crisply sweet, cocoa-toned. Lemon blossom, roa...","A gently spice-toned, floral- driven wet-proce...",This coffee tied for the third-highest rating ...
3,https://www.coffeereview.com/review/ethiopia-s...,92,Roast House,Ethiopia Suke Quto,"Spokane, Washington","Guji Zone, Oromia Region, south-central Ethiopia",Medium-Light,$19.00/16 ounces,2017-11-01,53/79,8.0,8.0,9.0,9.0,8.0,"Delicate, sweetly spice-toned. Pink peppercorn...",Lavender-like flowers and hints of zesty pink ...,This coffee tied for the third-highest rating ...
4,https://www.coffeereview.com/review/ethiopia-g...,94,Big Creek Coffee Roasters,Ethiopia Gedeb Halo Beriti,"Hamilton, Montana","Gedeb District, Gedeo Zone, southern Ethiopia",Medium,$16.50/12 ounces,2017-11-01,48/70,9.0,9.0,9.0,9.0,8.0,"Deeply sweet, subtly pungent. Honey, pear, tan...",A deeply and generously lush cup saved from se...,Southern Ethiopia coffees like this one are pr...


## Step 2: Fix Data Types

`review_date` is currently stored as text in "Month Year" format
(e.g., "November 2017"). Convert it to a proper date type so it can
be sorted, filtered, and stored correctly in SQL.